<a href="https://colab.research.google.com/github/Nicoley09/StreamLit-Checkpoint-2/blob/main/Streamlit_checkpoint_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# What You're Aiming For
In this checkpoint, we are going to work on the 'Financial Inclusion in Africa' dataset that was provided as part of the Financial Inclusion in Africa hosted by the Zindi platform.

Dataset description: The dataset contains demographic information and what financial services are used by approximately 33,600 individuals across East Africa. The ML model role is to predict which individuals are most likely to have or use a bank account.

The term financial inclusion means:  individuals and businesses have access to useful and affordable financial products and services that meet their needs – transactions, payments, savings, credit and insurance – delivered in a responsible and sustainable way.

➡️ Dataset link

https://i.imgur.com/UNUZ4zR.jpg

➡️Columns explanation


Instructions
Install the necessary packages
Import you data and perform basic data exploration phase
Display general information about the dataset
Create a pandas profiling reports to gain insights into the dataset
Handle Missing and corrupted values
Remove duplicates, if they exist
Handle outliers, if they exist
Encode categorical features
Based on the previous data exploration train and test a machine learning classifier
Create a streamlit application (locally) and add input fields for your features and a validation button at the end of the form
Import your ML model into the streamlit application and start making predictions given the provided features values
Deploy your application on Streamlit share:
Create a github and a streamlit share accounts
Create a new git repo
Upload your local code to the newly created git repo
log in to your streamlit account an deploy your application from the git repo


# Import Data and Explore

In [1]:
!pip install ydata-profiling
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ydata_profiling import ProfileReport

# Load dataset
df = pd.read_csv('Financial_inclusion_dataset.csv')

# Basic exploration
print(df.head())
print(df.info())
print(df.describe())
print(df.isnull().sum())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.7/398.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.3 MB/s eta 0:00:00
  country  year    uniqueid bank_account location_type cellphone_access  \
0   Kenya  2018  uniqueid_1          Yes         Rural              Yes   
1   Kenya  2018  uniqueid_2           No         Rural               No   
2   Kenya  2018  uniqueid_3          Yes         Urban              Yes   
3   Kenya  2018  uniqueid_4           No         Rural              Yes   
4   Kenya  2018  uniqueid_5           No         Urban               No   

   household_size  age_of_respondent gender_of_respondent  \
0          

# Generate a Profiling Report

In [2]:
profile = ProfileReport(df, title="Financial Inclusion Dataset Report", explorative=True)
profile.to_file("financial_inclusion_report.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 13/13 [00:01<00:00,  9.39it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# Handle Missing, Duplicates, and Outliers

In [3]:
# Remove duplicates
df = df.drop_duplicates()

# Fill missing values
df.fillna(df.median(numeric_only=True), inplace=True)  # numeric columns
df.fillna('Unknown', inplace=True)  # categorical columns

# Optional: handle outliers (example using IQR)
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5*IQR) & (df[col] <= Q3 + 1.5*IQR)]


# Encode Categorical Features

In [4]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=['object']).columns
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])


In [5]:
df.drop('uniqueid',axis= 1)

,country,year,bank_account,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,0,2018,1,0,1,3,24,0,5,2,3,9
1,0,2018,0,0,0,5,70,0,1,4,0,4
2,0,2018,1,1,1,5,26,1,3,3,5,9
3,0,2018,0,0,1,5,34,0,1,2,2,3
4,0,2018,0,1,0,8,26,1,0,3,2,5
...,...,...,...,...,...,...,...,...,...,...,...,...
23518,3,2018,0,0,1,9,20,0,0,3,2,6
23519,3,2018,0,0,1,4,48,0,1,0,0,7
23520,3,2018,0,0,1,2,27,0,1,3,3,7
23521,3,2018,0,0,1,5,27,0,4,4,2,7


# Split Data & Train ML Model

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Assuming 'bank_account' is the target
X = df.drop('bank_account', axis=1)
y = df['bank_account']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predictions and evaluation
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Save model
import joblib
joblib.dump(model, 'financial_inclusion_model.pkl')


Accuracy: 0.8698974023139052
              precision    recall  f1-score   support

           0       0.89      0.96      0.93      3899
           1       0.61      0.34      0.44       682

    accuracy                           0.87      4581
   macro avg       0.75      0.65      0.68      4581
weighted avg       0.85      0.87      0.85      4581



['financial_inclusion_model.pkl']

# Create Streamlit App

In [7]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# Load trained model
model = joblib.load('financial_inclusion_model.pkl')

st.set_page_config(page_title="Financial Inclusion Predictor", page_icon="💰", layout="centered")
st.title("💰 Financial Inclusion Predictor in East Africa")
st.write("Predict whether an individual is likely to have a bank account.")

st.sidebar.header("Enter Individual Details")

# --------------------
# Input fields
# --------------------
country = st.sidebar.selectbox("Country", [0,1,2])  # Adjust based on encoding
year = st.sidebar.number_input("Year", min_value=2000, max_value=2030, value=2022)
location_type = st.sidebar.selectbox("Location Type", ["Urban", "Rural"])
cellphone_access = st.sidebar.selectbox("Cellphone Access", ["No", "Yes"])
household_size = st.sidebar.number_input("Household Size", min_value=1, max_value=20, value=3)
age_of_respondent = st.sidebar.number_input("Age of Respondent", min_value=18, max_value=100, value=25)
gender_of_respondent = st.sidebar.selectbox("Gender", ["Male", "Female"])
relationship_with_head = st.sidebar.selectbox("Relationship with Head", ["Head","Spouse","Child","Other"])
marital_status = st.sidebar.selectbox("Marital Status", ["Single","Married","Divorced","Widowed"])
education_level = st.sidebar.selectbox("Education Level", ["None","Primary","Secondary","Tertiary"])
job_type = st.sidebar.selectbox("Job Type", ["Unemployed","Salaried","Self-employed","Other"])

# --------------------
# Encode categorical variables
# --------------------
location_map = {"Urban":0, "Rural":1}
cellphone_map = {"No":0, "Yes":1}
gender_map = {"Male":0, "Female":1}
relationship_map = {"Head":0,"Spouse":1,"Child":2,"Other":3}
marital_map = {"Single":0,"Married":1,"Divorced":2,"Widowed":3}
education_map = {"None":0,"Primary":1,"Secondary":2,"Tertiary":3}
job_map = {"Unemployed":0,"Salaried":1,"Self-employed":2,"Other":3}

# --------------------
# Create input DataFrame in exact order
# --------------------

# Example uniqueid placeholder
uniqueid = 1  # Can be any integer; just to match training columns

feature_columns = [
    "country","year","uniqueid","location_type","cellphone_access","household_size",
    "age_of_respondent","gender_of_respondent","relationship_with_head",
    "marital_status","education_level","job_type"
]

input_data = pd.DataFrame([[
    country,
    year,
    uniqueid,  # Added to match training features
    location_map[location_type],
    cellphone_map[cellphone_access],
    household_size,
    age_of_respondent,
    gender_map[gender_of_respondent],
    relationship_map[relationship_with_head],
    marital_map[marital_status],
    education_map[education_level],
    job_map[job_type]
]], columns=feature_columns)

# --------------------
# Prediction
# --------------------
if st.button("Predict"):
    prediction = model.predict(input_data)
    probability = model.predict_proba(input_data)[0][1]

    if prediction[0] == 1:
        st.success(f"✅ Likely to have a bank account (Probability: {probability:.2f})")
    else:
        st.error(f"❌ Unlikely to have a bank account (Probability: {probability:.2f})")


Writing app.py


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22902 entries, 0 to 23522
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   country                 22902 non-null  int64
 1   year                    22902 non-null  int64
 2   uniqueid                22902 non-null  int64
 3   bank_account            22902 non-null  int64
 4   location_type           22902 non-null  int64
 5   cellphone_access        22902 non-null  int64
 6   household_size          22902 non-null  int64
 7   age_of_respondent       22902 non-null  int64
 8   gender_of_respondent    22902 non-null  int64
 9   relationship_with_head  22902 non-null  int64
 10  marital_status          22902 non-null  int64
 11  education_level         22902 non-null  int64
 12  job_type                22902 non-null  int64
dtypes: int64(13)
memory usage: 2.4 MB
